In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_20newsgroups
data = fetch_20newsgroups(subset='train')

import pandas as pd
import re
import json
import contractions
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

lemmatizer = WordNetLemmatizer()

with open('slang.json', 'r') as f:
    slang_words = json.load(f)
stop_words = set(stopwords.words('english'))

In [14]:
class MultinomialNaiveBayes():
    def __init__(self, alpha = 1):
        self.classes_ = None
        self.n_classes_ = 0
        self.n_samples = 0
        self.dist = None
        self.priors_ = None
        self.feature_counts_ = None 
        self.class_feature_totals_ = None
        self.alpha = 1

    def fit(self, X, y): # CHECKED, matches with sklearn
        self.y = y
        self.n_samples, self.n_features = X.shape
        self.classes_ = np.unique(y)
        self.n_classes_ = self.classes_.shape[0]
        self.class_priors_ = self._prior()
        
        # distinct values in each features
        self.uniques = []
        for i in range(self.n_features):
            tmp = np.unique(X[:,i])
            self.uniques.append( tmp )
            
        self.N_yi = np.zeros((self.n_classes_, self.n_features)) # feature count
        self.N_y = np.zeros((self.n_classes_)) # total count 
        for i in self.classes_: # x axis
            indices = np.argwhere(self.y==i).flatten()
            columnwise_sum = []
            for j in range(self.n_features): # y axis
                columnwise_sum.append(np.sum(X[indices,j]))
                
            self.N_yi[i] = columnwise_sum # 2d
            self.N_y[i] = np.sum(columnwise_sum) # 1d
            
    def _prior(self): 
        P = np.zeros((self.n_classes_))
        _, self.dist = np.unique(self.y, return_counts=True)
        for i in range(self.n_classes_):
            P[i] = self.dist[i] / self.n_samples
        return P
    
    def _theta(self, x_i, i, h):
        
        Nyi = self.N_yi[h,i]
        Ny  = self.N_y[h]
        
        numerator = Nyi + self.alpha
        denominator = Ny + (self.alpha * self.n_features)
        
        return  (numerator / denominator)**x_i
    
    def _likelyhood(self, x, h):

        tmp = []
        for i in range(x.shape[0]):
            tmp.append(self._theta(x[i], i,h))
        
        return np.prod(tmp)
    
    def predict(self, X):
        samples, features = X.shape
        self.predict_proba = np.zeros((samples,self.n_classes_))
        
        for i in range(X.shape[0]):
            joint_likelyhood = np.zeros((self.n_classes_))
            
            for h in range(self.n_classes_):
                joint_likelyhood[h]  = self.class_priors_[h] * self._likelyhood(X[i],h) # P(y) P(X|y) 
                
            denominator = np.sum(joint_likelyhood)
            
            for h in range(self.n_classes_):
                numerator = joint_likelyhood[h]
                self.predict_proba[i,h] = (numerator / denominator)
            
        indices = np.argmax(self.predict_proba,axis=1)
        return self.classes_[indices]
def pipeline(X,y,X_test, y_test, alpha):
    """
    Sklearn Sanity Check
    """
    print("-"*20,'Sklearn',"-"*20)
    clf = MultinomialNB(alpha=alpha)#smooth parameter
    clf.fit(X,y)
    sk_y = clf.predict(X_test)
    print("Feature Count \n",clf.feature_count_)
    print("Class Log Prior ",clf.class_log_prior_)
    print('Accuracy ',accuracy_score(y_test, sk_y),sk_y)
    print(clf.predict_proba(X_test))
    print("-"*20,'Custom',"-"*20)
    nb = MultinomialNaiveBayes(alpha=alpha)#smooth parameter
    nb.fit(X,y)
    yhat = nb.predict(X_test)
    me_score = accuracy_score(y_test, yhat)
    print("Feature Count\n",nb.N_yi)
    print("Class Log Prior ",np.log(nb.class_priors_))
    print('Accuracy ',me_score,yhat)
    print(nb.predict_proba) # my predict proba is only for last test set

In [5]:
def flag_elongate(text):
    if not text: return text
    result = []
    for word in text.split():
        if re.search(r'(.)\1+', word):
            word = re.sub(r'(.)\1+', r'\1', word)
            word = f"{word} _elongated"
            result.append(word)
        else:
            result.append(word)
    return " ".join(result)

def flag_slang(text):
    if not text: return text
    result = []
    for word in text.split():
        if word in slang_words:
            result.append(slang_words[word] + " _slang") 
        else:
            result.append(word)
    return " ".join(result) 
    
def lemme(text):
    if not text: return text
    result = []
    for word in text.split():
        if word.startswith('_'):
            result.append(word)
        else:
            lemma = lemmatizer.lemmatize(word, pos='v')
            result.append(lemma)
    return " ".join(result) 
    
def remove_stopwords(text):
    if not text: return text
    result = []
    for word in text.split():
        if word not in stop_words:
            result.append(word)
    return " ".join(result)

def clean(text):
    # 1. Lowercase and Contractions
    lower = text.lower()
    text_expanded = contractions.fix(lower)
    
    # 2. Punctuation Spacing
    punc = []
    for ch in text_expanded:
        if (ch in string.punctuation) and (ch != "."):
            punc.append(f" {ch} ")
        else:
            punc.append(ch)
    punc = "".join(punc)

    # 3. Pipeline
    stop = remove_stopwords(punc)
    elong = flag_elongate(stop)
    sln = flag_slang(elong)
    lem = lemme(sln)
    
    return lem

In [6]:
df = pd.read_csv('data.csv')
#Clean data
# 1 flirt 0 not flirt
df.rename(columns={"polarity": "label", "final_messages": "text"},inplace=True)
df.dropna(inplace = True, ignore_index=True)
df.drop_duplicates(subset=["text"],inplace=True, ignore_index=True)
df['cleaned_text'] = df['text'].apply(clean)
print(df.to_string()) 

      label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 text                                                                                                                                                                                                                                        

In [7]:
count_neg = df['label'].value_counts().get(0, 0)
count_pos = df['label'].value_counts().get(1, 0)
delta = count_neg-count_pos

indices_to_drop = df[df['label'] == 0].sample(n=delta, random_state=42).index

df_balance = df.drop(indices_to_drop)

df_balance = df_balance.reset_index(drop=True)
print(df_balance.to_string()) 

      label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 text                                                                                                                                                                                                                                        

In [19]:
X = df_balance['cleaned_text']
y = df_balance['label']

vectorizer = TfidfVectorizer()
X_vectorized = vectorizer.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = MultinomialNB()
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
# print(f'Accuracy: {accuracy * 100:.2f}%')
# print(classification_report(y_test, y_pred))


y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Convert X_train and X_test to dense arrays for the custom pipeline
pipeline(X_train.toarray(), y_train, X_test=X_test.toarray(), y_test=y_test, alpha=10)

-------------------- Sklearn --------------------
Feature Count 
 [[4.77210385 0.56769284 0.         ... 1.18260675 0.42806459 0.        ]
 [3.00131814 0.         0.52937784 ... 1.30727628 0.         0.        ]]
Class Log Prior  [-0.70267106 -0.68371315]
Accuracy  0.7180094786729858 [1 0 1 0 1 1 1 1 1 1 0 0 0 0 0 1 0 1 1 1 1 1 1 0 0 0 1 0 1 1 1 0 1 0 1 0 0
 0 1 1 0 0 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 0 0 1 1 0 1 1
 1 0 0 1 0 1 1 1 0 1 1 0 1 0 1 0 1 1 1 1 0 0 0 1 1 1 1 0 1 1 1 1 1 0 0 0 0
 1 0 0 1 0 1 0 1 1 0 0 0 1 1 1 1 0 0 1 0 0 1 1 1 0 1 0 1 1 1 1 0 1 1 1 1 0
 1 1 1 1 1 1 0 1 0 0 1 1 1 0 1 1 1 1 1 0 1 1 1 1 0 0 0 1 0 0 1 0 1 1 1 1 1
 1 1 0 1 0 1 0 1 0 1 1 0 1 0 0 1 0 0 0 1 0 1 1 1 1 1 1 1 1 0 0 1 0 0 0 0 0
 0 1 1 0 1 1 1 1 0 0 1 0 1 0 1 0 0 1 1 0 0 1 1 1 0 1 0 1 1 1 0 0 1 0 1 0 0
 1 0 0 0 1 1 1 0 1 1 1 1 0 0 1 0 0 1 0 1 1 0 1 0 0 0 1 1 1 0 0 1 1 0 1 0 1
 0 1 0 0 1 0 1 1 0 1 1 1 0 1 1 1 1 0 1 1 0 1 0 1 1 1 1 1 1 0 0 0 1 1 1 1 0
 1 1 0 0 1 1 0 0 0 1 1 1 1 1 1 1 0 1 0 0

/Users/tunt/anaconda3/envs/AI_tech/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tunt/anaconda3/envs/AI_tech/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tunt/anaconda3/envs/AI_tech/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/tunt/anaconda3/envs/AI_tech/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tunt/anaconda3/envs/AI_tech/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tunt/anaconda3/envs/AI_tech/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/tunt/anaconda3/envs/AI_tech/lib/python3.1

Feature Count
 [[4.77210385 0.56769284 0.         ... 1.18260675 0.42806459 0.        ]
 [3.00131814 0.         0.52937784 ... 1.30727628 0.         0.        ]]
Class Log Prior  [-0.70267106 -0.68371315]
Accuracy  0.7251184834123223 [1 0 1 0 1 1 0 1 1 1 0 0 0 0 0 1 0 1 1 0 1 1 0 0 0 0 1 0 1 1 1 0 1 0 1 0 0
 0 1 1 0 0 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 0 1 0 0 1 1 0 1 1
 1 0 0 1 0 1 1 1 0 1 1 1 1 0 1 0 1 1 1 1 0 0 0 1 1 1 1 0 1 1 1 1 1 0 0 0 0
 0 0 0 1 0 1 0 1 1 0 0 0 1 1 0 1 0 0 1 0 0 1 1 1 0 1 0 1 1 1 1 0 1 1 1 1 0
 1 1 1 1 1 1 0 1 0 0 1 1 1 0 1 1 1 1 1 0 1 1 1 1 1 0 0 1 0 0 1 0 1 0 0 1 1
 1 1 0 1 0 1 0 1 0 1 1 0 1 0 0 1 0 0 0 1 0 1 1 1 1 1 1 1 1 0 0 1 0 0 0 0 0
 0 1 1 0 1 0 1 1 0 0 1 0 1 0 1 0 0 1 1 1 0 1 1 1 0 1 0 1 1 1 0 0 1 0 1 1 0
 1 0 0 0 1 1 1 0 1 1 1 0 0 0 1 0 0 0 0 1 1 0 1 0 0 0 1 1 1 0 0 1 1 0 1 0 1
 0 1 1 0 1 0 1 1 0 1 1 1 0 1 1 1 1 0 1 0 0 1 0 1 1 1 1 1 1 0 0 0 1 1 1 1 0
 1 1 0 0 1 1 0 0 0 1 1 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 1 1 1 0 1 1 0 0 1 0 0
 1 1 1 0 0 0 0 1

In [ ]:
def prior_prob():
    return
def likehood():
    
    return
def c_prob():
    return

P = np.zeros()